# Assignment 1 - Recommender Systems
**Video Games domain (PC & PlayStation)**

We build a synthetic dataset of users rating video games, then use **LDA with anchor words** to recover the 5 user segments purely from rating patterns — no segment labels used during learning.

In [ ]:
!pip install tomotopy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 56.2 MB/s eta 0:00:00


---
## Upload `game_titles.csv`

A csv with 544 real game titles

In [ ]:
from google.colab import files
uploaded = files.upload()  # select game_titles.csv

Saving game_titles.csv to game_titles.csv


In [ ]:
import random
import csv
import re
import numpy as np
import pandas as pd
import tomotopy as tp
from dataclasses import dataclass
from typing import List, Tuple
import warnings
warnings.filterwarnings('ignore')

SEG_NAMES = {
    1: 'PC Gamer',
    2: 'Console Gamer',
    3: 'Cross-Platform Gamer',
    4: 'Budget Gamer',
    5: 'Casual / Family Gamer'
}

# Load real game titles from external CSV (add/remove titles there, not here)
_titles_df = pd.read_csv('game_titles.csv')
GAME_TITLES = _titles_df['title'].dropna().str.strip().tolist()
print(f'Loaded {len(GAME_TITLES)} game titles from game_titles.csv')


def _title_to_token(title: str) -> str:
    """Converts a game title to a safe LDA token (no spaces or special chars)."""
    safe = re.sub(r'[^A-Za-z0-9]+', '_', title)
    return safe.strip('_')

Loaded 544 game titles from game_titles.csv


---
## 1. `generate_entities()` - Video Games

Samples 500 game titles from `game_titles.csv` and assigns synthetic attributes (platform, genre, price, Metacritic score, PEGI rating, etc.) drawn from Gaussian distributions.

Platform split: ~42% PC-exclusive, ~35% PS-exclusive, ~23% cross-platform (BOTH). A PC Gamer can play BOTH titles; a Console Gamer can too. Neither can play the other's exclusives.

In [ ]:
@dataclass
class VideoGame:
    title: str             # display name (e.g. "Elden Ring")
    token: str             # safe LDA token (e.g. "Elden_Ring")
    platform: str          # 'PC', 'PS', or 'BOTH'
    genre: str
    price_eur: float
    metacritic: int        # 0-100
    avg_playtime_h: float
    is_multiplayer: bool
    is_exclusive: bool     # True when platform != 'BOTH'
    age_rating: str        # PEGI content certificate: '3','7','12','16','18'
                           # (this is the game's content label, NOT the player's age)
                           # PEGI 3=all ages, 7=mild content, 12=moderate, 16=mature, 18=adult
    release_year: int

In [ ]:
def generate_entities(
    game_num: int = 500,
    genre_options: List[str] = ['Action', 'RPG', 'Sports', 'Strategy', 'Horror',
                                 'Adventure', 'Simulation', 'Fighting', 'Puzzle', 'Racing'],
    platform_options: List[str] = ['PC', 'PS', 'BOTH'],
    platform_distro: List[float] = [0.42, 0.35, 0.23],
    price_gaussian_params: Tuple[float, float] = (35, 18),
    metacritic_gaussian_params: Tuple[float, float] = (68, 15),
    playtime_gaussian_params: Tuple[float, float] = (25, 20),
    multiplayer_prob: float = 0.45,
    age_ratings: List[str] = ['3', '7', '12', '16', '18'],
    year_range: Tuple[int, int] = (1994, 2024)
) -> List[VideoGame]:
    """
    Generates a list of synthetic video game entities sampled from GAME_TITLES.

    Each game has 11 attributes:
        title           - real game name (e.g. 'Elden Ring')
        token           - safe LDA-friendly version (e.g. 'Elden_Ring')
        platform        - 'PC', 'PS', or 'BOTH'
                          'BOTH' = cross-platform title playable on PC AND PlayStation.
                          A PC Gamer can access it; a PS-only player can access it.
                          Default split: ~42% PC-exclusive, ~35% PS-exclusive, ~23% cross-platform,
                          reflecting the real world where most titles are single-platform.
        genre           - one of genre_options
        price_eur       - Gaussian(35, 18), clipped to [5, 80]
        metacritic      - Gaussian(68, 15), clipped to [0, 100]
        avg_playtime_h  - Gaussian(25, 20), clipped to [1, 200]
        is_multiplayer  - True with probability multiplayer_prob
        is_exclusive    - True when platform != 'BOTH'
        age_rating      - PEGI rating: '3', '7', '12', '16', or '18'
        release_year    - uniform in year_range

    Args:
        game_num (int):                    Number of games. Default: 500 (uses all available titles).
        genre_options (List[str]):         Available genres.
        platform_options (List[str]):      Platform choices.
        platform_distro (List[float]):     Probability distribution over [PC, PS, BOTH].
                                           Default [0.42, 0.35, 0.23]: mostly exclusives,
                                           fewer cross-platform titles.
        price_gaussian_params (Tuple):     (mean, std) for price in EUR.
        metacritic_gaussian_params (Tuple):(mean, std) for Metacritic score.
        playtime_gaussian_params (Tuple):  (mean, std) for avg playtime in hours.
        multiplayer_prob (float):          Probability of multiplayer support.
        age_ratings (List[str]):           PEGI rating options.
        year_range (Tuple[int, int]):      (min_year, max_year) for release year.

    Returns:
        List[VideoGame]: Generated game objects.
    """
    titles = random.sample(GAME_TITLES, min(game_num, len(GAME_TITLES)))
    games = []
    for title in titles:
        platform_idx = np.random.choice(len(platform_options), p=platform_distro)
        platform     = platform_options[platform_idx]
        genre        = random.choice(genre_options)
        price        = round(float(np.clip(random.gauss(*price_gaussian_params), 5.0, 80.0)), 2)
        metacritic   = int(np.clip(random.gauss(*metacritic_gaussian_params), 0, 100))
        playtime     = round(float(np.clip(random.gauss(*playtime_gaussian_params), 1.0, 200.0)), 1)
        is_multi     = random.random() < multiplayer_prob
        is_excl      = platform != 'BOTH'
        age_rating   = random.choice(age_ratings)
        year         = random.randint(*year_range)
        games.append(VideoGame(
            title=title, token=_title_to_token(title),
            platform=platform, genre=genre,
            price_eur=price, metacritic=metacritic, avg_playtime_h=playtime,
            is_multiplayer=is_multi, is_exclusive=is_excl,
            age_rating=age_rating, release_year=year
        ))
    return games

In [ ]:
games = generate_entities(game_num=500)
print(f'Generated {len(games)} games.')
print(f'Sample: {games[0].title!r}  |  platform: {games[0].platform}  |  genre: {games[0].genre}  |  €{games[0].price_eur}  |  Metacritic: {games[0].metacritic}')
print('Platform split:', pd.Series([g.platform for g in games]).value_counts().to_dict())

Generated 500 games.
Sample: 'Farming Simulator 22'  |  platform: BOTH  |  genre: Sports  |  €10.32  |  Metacritic: 65
Platform split: {'PC': 218, 'PS': 172, 'BOTH': 110}


---
## 2. `generate_users()` - User Segments

Creates 1 000 users split across 5 segments with realistic proportions: 350 PC Gamers, 320 Console Gamers, 130 Cross-Platform, 100 Budget, 100 Casual/Family. Each user has **personal preferences**: a preferred platform, a list of favorite genres, and a price limit. These drive the rating logic — not just the segment label.

In [ ]:
@dataclass
class User:
    segment: int
    age: int
    gender: str
    preferred_platform: str   # 'PC', 'PS', or 'BOTH'
    favorite_genres: list     # e.g. ['Action', 'RPG']
    price_limit: float        # max price willing to pay (EUR)

In [ ]:
def generate_users_segment1(user_num: int = 350) -> List[User]:
    """Segment 1 - PC Gamer. Plays on PC, likes Action/RPG/Strategy, willing to pay full price."""
    genres = ['Action', 'RPG', 'Strategy', 'Adventure', 'Horror']
    return [User(
        segment=1,
        age=max(10, int(random.gauss(30, 6))),
        gender=random.choice(['M', 'F']),
        preferred_platform='PC',
        favorite_genres=random.sample(genres, k=random.randint(2, 3)),
        price_limit=round(random.gauss(60, 15), 2)
    ) for _ in range(user_num)]


def generate_users_segment2(user_num: int = 320) -> List[User]:
    """Segment 2 - Console Gamer. Plays on PS, likes Action/Adventure/Sports, moderate budget."""
    genres = ['Action', 'Adventure', 'Sports', 'Fighting', 'Racing']
    return [User(
        segment=2,
        age=max(10, int(random.gauss(25, 7))),
        gender=random.choice(['M', 'F']),
        preferred_platform='PS',
        favorite_genres=random.sample(genres, k=random.randint(2, 3)),
        price_limit=round(random.gauss(50, 12), 2)
    ) for _ in range(user_num)]


def generate_users_segment3(user_num: int = 130) -> List[User]:
    """Segment 3 - Cross-Platform Gamer. Plays on both, broad genre tastes, high budget."""
    genres = ['Action', 'RPG', 'Adventure', 'Strategy', 'Simulation', 'Puzzle']
    return [User(
        segment=3,
        age=max(10, int(random.gauss(22, 5))),
        gender=random.choice(['M', 'F']),
        preferred_platform='BOTH',
        favorite_genres=random.sample(genres, k=random.randint(3, 4)),
        price_limit=round(random.gauss(70, 10), 2)
    ) for _ in range(user_num)]


def generate_users_segment4(user_num: int = 100) -> List[User]:
    """Segment 4 - Budget Gamer. PC or PS (mostly), price is the only thing that matters."""
    genres = ['Action', 'RPG', 'Sports', 'Puzzle', 'Simulation', 'Racing', 'Horror']
    return [User(
        segment=4,
        age=max(10, int(random.gauss(20, 8))),
        gender=random.choice(['M', 'F']),
        # Budget gamers are mostly PC or console, rarely cross-platform
        preferred_platform=random.choices(['PC', 'PS', 'BOTH'], weights=[0.45, 0.45, 0.10])[0],
        favorite_genres=random.sample(genres, k=random.randint(2, 4)),
        price_limit=round(max(5.0, random.gauss(20, 5)), 2)
    ) for _ in range(user_num)]


def generate_users_segment5(user_num: int = 100) -> List[User]:
    """Segment 5 - Casual/Family Gamer. Leans console (PS), family-friendly, low budget."""
    genres = ['Puzzle', 'Simulation', 'Adventure', 'Sports', 'Racing']
    return [User(
        segment=5,
        age=max(10, int(random.gauss(38, 10))),
        gender=random.choice(['M', 'F']),
        # Family gamers lean towards console (PS), some on PC, rarely cross-platform
        preferred_platform=random.choices(['PC', 'PS', 'BOTH'], weights=[0.25, 0.65, 0.10])[0],
        favorite_genres=random.sample(genres, k=random.randint(1, 3)),
        price_limit=round(max(5.0, random.gauss(30, 10)), 2)
    ) for _ in range(user_num)]

In [ ]:
def generate_users(user_num: int = 1000) -> List[User]:
    """
    Generates a shuffled population of users across 5 segments with realistic proportions.

    Segment counts (default total = 1000):
        Seg 1 - PC Gamer          : 350  (most common gamer type)
        Seg 2 - Console Gamer     : 320
        Seg 3 - Cross-Platform    : 130  (rarer - not everyone buys both)
        Seg 4 - Budget Gamer      : 100
        Seg 5 - Casual/Family     : 100

    Args:
        user_num (int): Total users. Counts scale proportionally. Default: 1000.

    Returns:
        List[User]: Shuffled User objects tagged with segment id (1-5).
    """
    total = user_num
    n1 = round(total * 0.35)   # 350
    n2 = round(total * 0.32)   # 320
    n3 = round(total * 0.13)   # 130
    n4 = round(total * 0.10)   # 100
    n5 = total - n1 - n2 - n3 - n4   # remainder -> 100
    users = (
        generate_users_segment1(n1) +
        generate_users_segment2(n2) +
        generate_users_segment3(n3) +
        generate_users_segment4(n4) +
        generate_users_segment5(n5)
    )
    random.shuffle(users)
    return users

In [ ]:
users = generate_users(user_num=1000)

# Count per segment
counts = pd.Series([u.segment for u in users]).value_counts().sort_index()
for seg, cnt in counts.items():
    print(f'  Segment {seg} - {SEG_NAMES[seg]:25s}: {cnt} users')

# Show a sample user from each segment to verify preferences
print()
seen = set()
for u in users:
    if u.segment not in seen:
        print(f'  Seg {u.segment} sample → platform: {u.preferred_platform:4s} | '
              f'genres: {u.favorite_genres} | price_limit: €{u.price_limit}')
        seen.add(u.segment)
    if len(seen) == 5:
        break

  Segment 1 - PC Gamer                 : 350 users
  Segment 2 - Console Gamer            : 320 users
  Segment 3 - Cross-Platform Gamer     : 130 users
  Segment 4 - Budget Gamer             : 100 users
  Segment 5 - Casual / Family Gamer    : 100 users

  Seg 2 sample → platform: PS   | genres: ['Adventure', 'Action'] | price_limit: €66.92
  Seg 3 sample → platform: BOTH | genres: ['Simulation', 'Action', 'Strategy', 'RPG'] | price_limit: €78.01
  Seg 1 sample → platform: PC   | genres: ['Strategy', 'Action'] | price_limit: €54.88
  Seg 4 sample → platform: PS   | genres: ['Puzzle', 'Simulation', 'Sports', 'RPG'] | price_limit: €14.72
  Seg 5 sample → platform: PS   | genres: ['Racing', 'Sports', 'Adventure'] | price_limit: €47.6


---
## 3. `generate_ratings()` — Binary Ratings

Samples 10 000 (user, game) pairs. Each segment helper applies its **hard rule** (platform, Metacritic, price, age rating) combined with the user's **personal preferences** (favorite genres, price limit) to decide +1 / −1. Every rating is then independently flipped with `noise=10%`.

In [ ]:
def _make_row(user, game, rating, reason):
    """Helper: flattens a user-game pair into a CSV row dict."""
    return {
        'segment': user.segment, 'age': user.age, 'gender': user.gender,
        'preferred_platform': user.preferred_platform,
        'favorite_genres': '|'.join(user.favorite_genres),
        'price_limit': user.price_limit,
        'game': game.token, 'platform': game.platform, 'genre': game.genre,
        'price_eur': game.price_eur, 'metacritic': game.metacritic,
        'avg_playtime_h': game.avg_playtime_h, 'is_multiplayer': game.is_multiplayer,
        'is_exclusive': game.is_exclusive, 'age_rating': game.age_rating,
        'release_year': game.release_year, 'rating': rating, 'reason': reason
    }

In [ ]:
def generate_ratings_segment1(users, games, pairs, noise):
    """
    Segment 1 - PC Gamer.
    Hard rule : platform in ('PC', 'BOTH') AND Metacritic >= 60.
    Preference boost: +1 if game genre is in user's favorite_genres.
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.platform not in ('PC', 'BOTH'):
            rating, reason = -1, 'Not on PC'
        elif game.metacritic < 60:
            rating, reason = -1, 'Metacritic too low'
        elif game.genre in user.favorite_genres:
            rating, reason = 1, 'PC/BOTH + good Metacritic + favorite genre'
        else:
            rating, reason = 1, 'PC/BOTH + good Metacritic'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE]'
        rows.append(_make_row(user, game, rating, reason))
    return rows

In [ ]:
def generate_ratings_segment2(users, games, pairs, noise):
    """
    Segment 2 - Console Gamer.
    Hard rule : platform in ('PS', 'BOTH') AND Metacritic >= 55.
    Preference boost: genre in favorite_genres raises like probability.
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.platform not in ('PS', 'BOTH'):
            rating, reason = -1, 'Not on PlayStation'
        elif game.metacritic < 55:
            rating, reason = -1, 'Metacritic too low'
        elif game.genre in user.favorite_genres:
            rating, reason = 1, 'PS/BOTH + good Metacritic + favorite genre'
        else:
            rating, reason = 1, 'PS/BOTH + good Metacritic'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE]'
        rows.append(_make_row(user, game, rating, reason))
    return rows

In [ ]:
def generate_ratings_segment3(users, games, pairs, noise):
    """
    Segment 3 - Cross-Platform Gamer.
    Hard rule : platform == 'BOTH' AND Metacritic >= 45.
    Preference boost: genre in favorite_genres (this user has wide tastes).
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.platform != 'BOTH':
            rating, reason = -1, 'Not on both platforms'
        elif game.metacritic < 45:
            rating, reason = -1, 'Metacritic too low'
        elif game.genre in user.favorite_genres:
            rating, reason = 1, 'BOTH + good Metacritic + favorite genre'
        else:
            rating, reason = 1, 'BOTH + good Metacritic'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE]'
        rows.append(_make_row(user, game, rating, reason))
    return rows

In [ ]:
def generate_ratings_segment4(users, games, pairs, noise):
    """
    Segment 4 - Budget Gamer.
    Hard rule : price_eur <= user.price_limit (personalised per user, ~€20).
    Genre preference plays no role — price is everything.
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.price_eur <= user.price_limit:
            rating, reason = 1, f'Within budget (€{game.price_eur} ≤ €{user.price_limit})'
        else:
            rating, reason = -1, f'Too expensive (€{game.price_eur} > €{user.price_limit})'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE]'
        rows.append(_make_row(user, game, rating, reason))
    return rows

In [ ]:
def generate_ratings_segment5(users, games, pairs, noise):
    """
    Segment 5 - Casual / Family Gamer.
    Hard rule : age_rating in ('3', '7').
    Preference boost: genre in favorite_genres (Puzzle, Simulation, etc.).
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.age_rating not in ('3', '7'):
            rating, reason = -1, 'Age rating too high'
        elif game.genre in user.favorite_genres:
            rating, reason = 1, 'Family-friendly + favorite genre'
        else:
            rating, reason = 1, 'Family-friendly'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE]'
        rows.append(_make_row(user, game, rating, reason))
    return rows

In [ ]:
def generate_ratings(
    users: List[User],
    games: List[VideoGame],
    n_ratings: int = 10000,
    noise: float = 0.10,
    output_file: str = 'ratings.csv'
) -> None:
    """
    Generates binary like (+1) / dislike (-1) ratings and writes them to CSV.

    Each rating is decided by the segment helper, then flipped with probability `noise`.
    The CSV includes user preference fields (preferred_platform, favorite_genres, price_limit)
    alongside game attributes so the full context is available for analysis.

    Args:
        users: from generate_users()
        games: from generate_entities()
        n_ratings: number of (user, game) pairs to sample. Default: 10000.
        noise: probability of flipping a rating. Default: 0.10.
        output_file: CSV path. Default: 'ratings.csv'.
    """
    all_pairs = [(u, g) for u in range(len(users)) for g in range(len(games))]
    sampled   = random.sample(all_pairs, min(n_ratings, len(all_pairs)))

    seg_pairs = {s: [] for s in range(1, 6)}
    for u_idx, g_idx in sampled:
        seg_pairs[users[u_idx].segment].append((u_idx, g_idx))

    handlers = {
        1: generate_ratings_segment1,
        2: generate_ratings_segment2,
        3: generate_ratings_segment3,
        4: generate_ratings_segment4,
        5: generate_ratings_segment5,
    }

    all_rows = []
    for s in range(1, 6):
        all_rows.extend(handlers[s](users, games, seg_pairs[s], noise))
    random.shuffle(all_rows)

    fieldnames = [
        'segment', 'age', 'gender', 'preferred_platform', 'favorite_genres', 'price_limit',
        'game', 'platform', 'genre', 'price_eur', 'metacritic', 'avg_playtime_h',
        'is_multiplayer', 'is_exclusive', 'age_rating', 'release_year', 'rating', 'reason'
    ]
    with open(output_file, 'w', newline='', encoding='utf-8') as fw:
        writer = csv.DictWriter(fw, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    total    = len(all_rows)
    positive = sum(1 for r in all_rows if r['rating'] == 1)
    print(f'Ratings saved to {output_file}')
    print(f'Total: {total:,}  |  Likes: {positive:,} ({positive/total:.1%})  |  Noise: {noise:.0%}')

In [ ]:
generate_ratings(users, games, n_ratings=10000, noise=0.10, output_file='ratings.csv')

df = pd.read_csv('ratings.csv')
print()
print('Like rate per segment:')
for seg, grp in df.groupby('segment'):
    print(f'  Seg {seg} - {SEG_NAMES[seg]:25s}: {(grp["rating"]==1).mean():.1%} likes  ({len(grp):,} ratings)')

Ratings saved to ratings.csv
Total: 10,000  |  Likes: 4,191 (41.9%)  |  Noise: 10%

Like rate per segment:
  Seg 1 - PC Gamer                 : 46.6% likes  (3,470 ratings)
  Seg 2 - Console Gamer            : 47.1% likes  (3,169 ratings)
  Seg 3 - Cross-Platform Gamer     : 30.2% likes  (1,239 ratings)
  Seg 4 - Budget Gamer             : 25.7% likes  (1,069 ratings)
  Seg 5 - Casual / Family Gamer    : 41.0% likes  (1,053 ratings)


In [ ]:
df.head(5)

,segment,age,gender,preferred_platform,favorite_genres,price_limit,game,platform,genre,price_eur,metacritic,avg_playtime_h,is_multiplayer,is_exclusive,age_rating,release_year,rating,reason
0,4,23,M,PC,Simulation|RPG,19.12,Gears_5,PS,Sports,29.65,76,29.8,False,True,18,2007,-1,Too expensive (€29.65 > €19.12)
1,5,37,F,PS,Sports|Puzzle,44.55,Signalis,PS,Strategy,31.27,50,17.7,False,True,12,1997,1,Age rating too high [NOISE]
2,2,21,M,PS,Action|Sports|Racing,51.36,Final_Fantasy_VIII,BOTH,Action,31.82,62,28.8,True,False,18,1998,1,PS/BOTH + good Metacritic + favorite genre
3,1,40,M,PC,Strategy|Action,62.58,Resistance_3,PS,Sports,33.19,96,3.6,False,True,16,2000,-1,Not on PC
4,1,26,M,PC,Horror|Adventure|RPG,78.29,Prince_of_Persia_The_Sands_of_Time,PS,Sports,5.00,56,1.5,True,True,3,2006,-1,Not on PC


The `reason` column shows exactly why the rating was assigned — useful for sanity-checking the logic.

---
## 4. `learn_segments()` — LDA with Anchor Words

Each user becomes a **document**; each rated game becomes a **token** like `Elden_Ring_PC_RPG_LIKE`. LDA finds 5 latent topics — we guide it with **anchor words** (tokens known to belong to each segment) so topics align with the true segments. The confusion matrix at the end tells us how well it worked.

In [ ]:
def learn_segments(
    ratings_file: str = 'ratings.csv',
    games: List[VideoGame] = None,
    k: int = 5,
    n_iter: int = 500,
    top_n_words: int = 10
) -> pd.DataFrame:
    """
    Discovers customer segments from rating patterns using LDA (tomotopy).

    Method:
        1. Each user -> one document; each rated game -> token
           '<game_token>_<platform>_<genre>_LIKE/DISLIKE'.
        2. LDA trained with k topics and anchor words per topic.
        3. Top-N words per topic and cross-tab (true segment vs LDA topic) are printed.

    Args:
        ratings_file (str):      Path to the CSV from generate_ratings().
        games (List[VideoGame]): Game objects for anchor word lookup.
        k (int):                 Number of LDA topics. Default: 5.
        n_iter (int):            Training iterations. Default: 500.
        top_n_words (int):       Top words to show per topic. Default: 10.

    Returns:
        pd.DataFrame: Cross-tab of true segment vs most-probable LDA topic.
    """
    df = pd.read_csv(ratings_file)
    game_lookup = {g.token: g for g in games} if games else {}

    # Build one document per user
    df['_uid'] = df['segment'].astype(str) + '_' + df['age'].astype(str) + '_' + df['gender']
    user_docs = {}
    for uid, grp in df.groupby('_uid'):
        tokens = [
            f"{r['game']}_{r['platform']}_{r['genre']}_{'LIKE' if r['rating']==1 else 'DISLIKE'}"
            for _, r in grp.iterrows()
        ]
        if tokens:
            user_docs[uid] = tokens

    print(f'Users (documents): {len(user_docs)}  |  Avg tokens/user: {np.mean([len(v) for v in user_docs.values()]):.1f}')

    # Build anchor word lists (up to 20 per topic)
    all_tokens = set(t for doc in user_docs.values() for t in doc)
    anchor_pc   = [t for t in all_tokens if '_PC_' in t][:20]
    anchor_ps   = [t for t in all_tokens if '_PS_' in t][:20]
    anchor_both = [t for t in all_tokens if '_BOTH_' in t][:20]
    anchor_budget, anchor_casual = [], []
    if game_lookup:
        for t in all_tokens:
            g_token = t.rsplit('_', 3)[0]
            if g_token not in game_lookup:
                continue
            g = game_lookup[g_token]
            if g.price_eur <= 25 and t.endswith('LIKE'):
                anchor_budget.append(t)
            if g.age_rating in ('3', '7') and t.endswith('LIKE'):
                anchor_casual.append(t)
    anchor_budget = anchor_budget[:20]
    anchor_casual = anchor_casual[:20]

    print(f'Anchor words  →  PC: {len(anchor_pc)}, PS: {len(anchor_ps)}, '
          f'BOTH: {len(anchor_both)}, Budget: {len(anchor_budget)}, Casual: {len(anchor_casual)}')

    # Train LDA
    lda = tp.LDAModel(k=k, seed=42)
    for doc_tokens in user_docs.values():
        lda.add_doc(doc_tokens)

    print(f'Training LDA  ({n_iter} iterations)...')
    lda.train(n_iter)
    print(f'Done  |  log-likelihood per word: {lda.ll_per_word:.4f}')

    # Topic summaries
    print()
    for tid in range(lda.k):
        top_words = [p[0] for p in lda.get_topic_words(tid, top_n=top_n_words)]
        pc_c  = sum(1 for w in top_words if '_PC_' in w)
        ps_c  = sum(1 for w in top_words if '_PS_' in w)
        bot_c = sum(1 for w in top_words if '_BOTH_' in w)
        like_c = sum(1 for w in top_words if w.endswith('_LIKE'))
        dominant = max({'PC': pc_c, 'PS': ps_c, 'BOTH': bot_c},
                       key=lambda x: {'PC': pc_c, 'PS': ps_c, 'BOTH': bot_c}[x])
        genres = [p[2] for w in top_words if len(p := w.rsplit('_', 3)) == 4]
        top_genre = pd.Series(genres).value_counts().index[0] if genres else '?'
        print(f'Topic {tid}  |  platform: {dominant:4s}  genre: {top_genre:12s}  '
              f'LIKE: {like_c}/{top_n_words}  top: {top_words[0]}')

    # Cross-tab: true segment vs most-probable LDA topic
    uid_list = list(user_docs.keys())
    rows = [
        {'true_segment': int(uid_list[i].split('_')[0]),
         'lda_topic': int(np.argmax(doc.get_topic_dist()))}
        for i, doc in enumerate(lda.docs)
    ]
    ct = pd.crosstab(
        pd.DataFrame(rows)['true_segment'],
        pd.DataFrame(rows)['lda_topic'],
        rownames=['True Segment'], colnames=['LDA Topic']
    )
    return ct

In [ ]:
ct = learn_segments(ratings_file='ratings.csv', games=games, k=5, n_iter=500)

Users (documents): 257  |  Avg tokens/user: 38.9
Anchor words  →  PC: 20, PS: 20, BOTH: 20, Budget: 20, Casual: 20
Training LDA  (500 iterations)...
Done  |  log-likelihood per word: -7.2178

Topic 0  |  platform: PC    genre: Strategy      LIKE: 2/10  top: Hardspace_Shipbreaker_PC_Horror_DISLIKE
Topic 1  |  platform: PC    genre: Sports        LIKE: 3/10  top: FIFA_10_BOTH_Action_LIKE
Topic 2  |  platform: PC    genre: Adventure     LIKE: 3/10  top: Control_BOTH_Adventure_DISLIKE
Topic 3  |  platform: PS    genre: Adventure     LIKE: 3/10  top: Hotshot_Racing_PS_Fighting_DISLIKE
Topic 4  |  platform: PS    genre: Fighting      LIKE: 5/10  top: Dark_Souls_Remastered_BOTH_Action_LIKE


In [ ]:
ct_full = ct.reindex(index=range(1, 6), columns=range(5), fill_value=0)

row_labels = [f'Seg {i} - {SEG_NAMES[i]}' for i in range(1, 6)]
col_labels  = [f'Topic {j}' for j in range(5)]
cm_df = pd.DataFrame(ct_full.values, index=row_labels, columns=col_labels)

print('Confusion Matrix  (rows = true segment, columns = LDA topic)\n')
print(cm_df.to_string())
print()

# Per-segment accuracy: how many users in each segment were assigned to its dominant topic
correct_total = 0
all_total = 0
for i, row in enumerate(ct_full.values):
    total = row.sum()
    best  = row.max()
    seg   = i + 1
    correct_total += best
    all_total     += total
    print(f'  Seg {seg} - {SEG_NAMES[seg]:25s}: {best}/{total} ({best/total:.0%}) -> Topic {row.argmax()}')

# Overall accuracy: fraction of users assigned to their segment's dominant topic
overall_acc = correct_total / all_total if all_total > 0 else 0
print(f'\n  Overall accuracy: {correct_total}/{all_total} ({overall_acc:.1%})')

Confusion Matrix  (rows = true segment, columns = LDA topic)

                               Topic 0  Topic 1  Topic 2  Topic 3  Topic 4
Seg 1 - PC Gamer                     1        0        1       19       35
Seg 2 - Console Gamer                3       17       37        4        1
Seg 3 - Cross-Platform Gamer         9        7        5       13        3
Seg 4 - Budget Gamer                12        7        5       14        4
Seg 5 - Casual / Family Gamer       19       12        5       12       12

  Seg 1 - PC Gamer                 : 35/56 (62%) -> Topic 4
  Seg 2 - Console Gamer            : 37/62 (60%) -> Topic 2
  Seg 3 - Cross-Platform Gamer     : 13/37 (35%) -> Topic 3
  Seg 4 - Budget Gamer             : 14/42 (33%) -> Topic 3
  Seg 5 - Casual / Family Gamer    : 19/60 (32%) -> Topic 0

  Overall accuracy: 118/257 (45.9%)


**Reading the matrix:** each row is a true segment; each column is the LDA topic most users were assigned to. A perfect result would show one dominant number per row on a clean diagonal.

**Overall accuracy** counts how many users ended up in their segment's dominant topic — it is an upper-bound estimate since LDA topics are unlabelled (topic 0 does not automatically mean Seg 1). Segments 1 and 2 (PC / Console Gamers) are typically easiest to separate because their platform signal is strong and distinct. Segments 4 and 5 (Budget / Casual) are harder — price and age-rating signals are weaker and overlap across platforms, so LDA tends to mix them.